# Manager Evaluation

Compares four strategies over the same period and set of stocks:

| Strategy | Description |
|---|---|
| **Manager v1** | Fundamental analysis only |
| **Manager v2** | Fundamental analysis + material facts |
| **Baseline** | Price-only manager |
| **Buy & Hold** | Benchmark: buy on day 1, hold until last day |

In [8]:
import json
import numpy as np
from tabulate import tabulate

RESULTS_V1 = "results/manager"
RESULTS_V2 = "results/manager_v2"
RESULTS_BASE = "results/baseline"

## Helper Functions

In [9]:
def compute_portfolio_returns(decisions: list, results: list) -> dict:
    """Simulate portfolio trades and return realized trade returns per stock.

    For each decision (sorted by analysis_date):
    - 'Comprar' opens (or adds to) a position at the day's closing price.
    - 'Vender' closes all open positions; trade return = (sell - avg_buy) / avg_buy.
    Remaining open positions are closed at the last observed price.

    Returns
    -------
    dict[str, list[float]]
        Realized trade returns per stock ticker.
    """
    # Build a lookup: (stock_id, date) -> closing price
    price_lookup = {(r["ACAO"], r["DATA_DO_PREGAO"]): r["PRECO_ULTIMO_NEGOCIO"] for r in results}

    open_positions: dict[str, list] = {}
    trade_returns: dict[str, list] = {}
    last_price: dict[str, float] = {}

    for decision in decisions:
        stock_id = decision["stock_id"]
        date = decision["analysis_date"]
        rec = decision["recommendation"]
        price = price_lookup[(stock_id, date)]

        last_price[stock_id] = price
        open_positions.setdefault(stock_id, [])
        trade_returns.setdefault(stock_id, [])

        if rec == "Comprar":
            open_positions[stock_id].append(price)
        elif rec == "Vender" and open_positions[stock_id]:
            avg_buy = np.mean(open_positions[stock_id])
            trade_returns[stock_id].append((price - avg_buy) / avg_buy)
            open_positions[stock_id] = []

    # Close any remaining open positions at last known price
    for stock_id, positions in open_positions.items():
        if positions:
            avg_buy = np.mean(positions)
            trade_returns[stock_id].append((last_price[stock_id] - avg_buy) / avg_buy)

    return trade_returns

In [10]:
def compute_buy_and_hold(results: list) -> dict:
    """Compute buy-and-hold return per stock.

    Buys at the first observed closing price and sells at the last.

    Returns
    -------
    dict[str, float]
        Total return per stock ticker.
    """
    prices_per_stock: dict[str, list] = {}
    for r in results:
        stock_id = r["ACAO"]
        prices_per_stock.setdefault(stock_id, [])
        prices_per_stock[stock_id].append(r["PRECO_ULTIMO_NEGOCIO"])

    return {
        stock_id: (prices[-1] - prices[0]) / prices[0]
        for stock_id, prices in prices_per_stock.items()
        if len(prices) >= 2
    }


def total_return(trade_returns: dict) -> float:
    """Sum all realized trade returns across all stocks."""
    return sum(sum(v) for v in trade_returns.values())


def total_bnh_return(bnh: dict) -> float:
    """Sum buy-and-hold returns across all stocks."""
    return sum(bnh.values())

## Load Data

In [11]:
def load(folder: str) -> tuple:
    with open(f"{folder}/decisions_sample.json") as f:
        decisions = json.load(f)
    with open(f"{folder}/results_sample.json") as f:
        results = json.load(f)
    return decisions, results


decisions_v1, results_v1 = load(RESULTS_V1)
decisions_v2, results_v2 = load(RESULTS_V2)
decisions_base, results_base = load(RESULTS_BASE)

print(f"Manager v1 : {len(decisions_v1)} decisions, {len(results_v1)} price records")
print(f"Manager v2 : {len(decisions_v2)} decisions, {len(results_v2)} price records")
print(f"Baseline   : {len(decisions_base)} decisions, {len(results_base)} price records")

Manager v1 : 119 decisions, 120 price records
Manager v2 : 119 decisions, 120 price records
Baseline   : 120 decisions, 120 price records


## Compute Returns

In [12]:
returns_v1 = compute_portfolio_returns(decisions_v1, results_v1)
returns_v2 = compute_portfolio_returns(decisions_v2, results_v2)
returns_base = compute_portfolio_returns(decisions_base, results_base)
bnh = compute_buy_and_hold(results_v1)

## Per-Stock Comparison

In [13]:
all_stocks = sorted(set(returns_v1) | set(returns_v2) | set(returns_base) | set(bnh))

headers = ["Stock", "Manager v1", "Manager v2 (mat. facts)", "Baseline", "Buy & Hold"]
rows = []
for stock in all_stocks:
    r1 = sum(returns_v1.get(stock, [0.0]))
    r2 = sum(returns_v2.get(stock, [0.0]))
    rb = sum(returns_base.get(stock, [0.0]))
    rbh = bnh.get(stock, 0.0)
    rows.append(
        [
            stock,
            f"{r1 * 100:.2f}%",
            f"{r2 * 100:.2f}%",
            f"{rb * 100:.2f}%",
            f"{rbh * 100:.2f}%",
        ]
    )

# TOTAL row
rows.append(
    [
        "**TOTAL**",
        f"{total_return(returns_v1) * 100:.2f}%",
        f"{total_return(returns_v2) * 100:.2f}%",
        f"{total_return(returns_base) * 100:.2f}%",
        f"{total_bnh_return(bnh) * 100:.2f}%",
    ]
)

print(tabulate(rows, headers=headers, tablefmt="github"))

| Stock     | Manager v1   | Manager v2 (mat. facts)   | Baseline   | Buy & Hold   |
|-----------|--------------|---------------------------|------------|--------------|
| ALUP11    | 4.63%        | 16.12%                    | 17.99%     | 12.25%       |
| AURE3     | 18.75%       | 33.44%                    | 24.34%     | -8.12%       |
| CPLE3     | 25.34%       | 42.88%                    | 0.00%      | 46.87%       |
| ENEV3     | 115.18%      | 58.32%                    | 33.38%     | 54.63%       |
| EQTL3     | 20.08%       | 47.08%                    | 0.00%      | 12.29%       |
| **TOTAL** | 183.98%      | 197.84%                   | 75.71%     | 117.91%      |


## Aggregate Summary

In [14]:
summary_headers = ["Strategy", "Total Return"]
summary_rows = [
    ["Manager v1 (fundamental only)", f"{total_return(returns_v1) * 100:.2f}%"],
    ["Manager v2 (fundamental + mat. facts)", f"{total_return(returns_v2) * 100:.2f}%"],
    ["Baseline (price only)", f"{total_return(returns_base) * 100:.2f}%"],
    ["Buy & Hold", f"{total_bnh_return(bnh) * 100:.2f}%"],
]
print(tabulate(summary_rows, headers=summary_headers, tablefmt="github"))

| Strategy                              | Total Return   |
|---------------------------------------|----------------|
| Manager v1 (fundamental only)         | 183.98%        |
| Manager v2 (fundamental + mat. facts) | 197.84%        |
| Baseline (price only)                 | 75.71%         |
| Buy & Hold                            | 117.91%        |
